## Stage 2 continued — Finding 09 (per-layer sweep) + Finding 10 (probe readout)

Follow-up to finding 08 (patching v2): AGE (faithful map) → the bridge is
connected at layer 11. RACE (fragile map) → NOT connected at layer 11;
instead the 1-star-head pushes the wrong way. Two questions remain:

- **Finding 09** — if not L11, where does RACE answer from? Sweep all
  32 layers (all 32 heads per layer, one layer per condition) → an
  influence-on-output map, later paired up (locally) with the fidelity map
  (finding 06).
- **Finding 10** — how much information is "lost in the corridor"?
  Train a small probe reading L11 (all heads + star head only) at the
  OPINION ANSWER token position (not the identity token) → compare the
  probe's accuracy vs the mouth's accuracy, both against `group_real_dist`.
  Probe fitting is done LOCALLY (CPU) from the features extracted here --
  the same pattern as notebook 10.

**Efficiency trick (lesson from notebook 12):** a sweep of 32 layers + 1
random control = 33 conditions per (pair, question). Instead of 33 forward
passes one at a time, all of them are merged into **1 batch** -- each item
in that batch is patched at a DIFFERENT single layer simultaneously
(verified offline via a dry-run before this notebook was written: the
per-item patches within one batch are isolated, they do not leak into
other items).


## Before running: Kaggle setup

1. **Accelerator**: GPU T4 x2. **Internet: On**. Attach `opinionqa_intersectional.csv`.
2. Session left over from a crash -> RESTART SESSION.
3. **Download when finished** from `/kaggle/working/stage2_sweep_probe/`:
   `sweep_rows.csv`, `sweep_summary.csv`, `probe_features_L11.csv`,
   `probe_features_L11.npz` -> put them in
   `results/07_sweep_and_probe/`.

Estimate: model load ~10 min; batched baseline ~10-15 min; sweep
(720 batched forwards, batch=33) ~15-20 min. Total ~45 min - 1 hour.


In [ ]:
!pip install -q -U "transformers>=4.44" accelerate scipy tqdm

In [ ]:
import os, sys, gc, glob, ast
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from scipy.stats import wasserstein_distance, wilcoxon
from tqdm.auto import tqdm

sys.last_traceback = None
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    for d in range(torch.cuda.device_count()):
        free, total = torch.cuda.mem_get_info(d)
        print(f"GPU {d}: {free/1e9:.1f} GB free / {total/1e9:.1f} GB total")
        if free / total < 0.9:
            print(f"  WARNING: GPU {d} is not empty -> RESTART SESSION first!")


In [ ]:
MODEL_PATH = "mistralai/Mistral-7B-v0.1"

_candidates = glob.glob("/kaggle/input/**/opinionqa_intersectional.csv", recursive=True)
if _candidates:
    DATA_PATH = _candidates[0]
elif os.path.exists("opinionqa_intersectional.csv"):
    DATA_PATH = "opinionqa_intersectional.csv"
else:
    raise FileNotFoundError("opinionqa_intersectional.csv not found.")
print("Data:", DATA_PATH)

RANDOM_SEED = 42
TYPES_RUN = ["AGExPOLPARTY", "RELIGxPOLPARTY", "RACExRELIG"]  # same as notebook 12
N_PAIRS = 12
N_QUESTIONS = 20      # per pair, top by largest real WD
MAX_OPTIONS = 6
MIN_SHARED_Q = 20

STAR_LAYER, STAR_HEAD = 11, 16   # reference for finding 10 (single-head probe)
_forbidden_ctrl = {(STAR_LAYER, STAR_HEAD)}

OUT_DIR = "/kaggle/working/stage2_sweep_probe"
os.makedirs(OUT_DIR, exist_ok=True)


## 1. Data: pairs + questions with the largest real disagreement

Exactly the notebook 12 logic (questions picked PER PAIR, top-N by the WD
of the REAL distributions -- coverage across cells is sparse, RACExRELIG
has no question that every cell answers).


In [ ]:
df = pd.read_csv(DATA_PATH)
for c in ["responses", "ordinal", "options"]:
    df[c] = df[c].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
df["group_key"] = df["attribute"] + " :: " + df["group"]
df["n_opt"] = df["ordinal"].apply(len)

qmeta = {}
real_resp = {}
for r in df.itertuples():
    qmeta[r.qkey] = (r.question, r.options[: r.n_opt], r.ordinal)
    real_resp[(r.group_key, r.qkey)] = np.array(r.responses, dtype=np.float64)

def real_wd(A, B, qk):
    _, _, ordinal = qmeta[qk]
    return wasserstein_distance(ordinal, ordinal,
                                u_weights=real_resp[(A, qk)], v_weights=real_resp[(B, qk)])

rng = np.random.default_rng(RANDOM_SEED)
plan = {}
for ty in TYPES_RUN:
    sub = df[(df["attribute"] == ty) & (df["n_opt"] <= MAX_OPTIONS)]
    cells = sorted(sub["group_key"].unique().tolist())
    q_per_cell = sub.groupby("group_key")["qkey"].apply(set).to_dict()
    v1_opts = sorted({gk.split(" :: ", 1)[1].split(" | ", 1)[0] for gk in cells})
    v2_opts = sorted({gk.split(" :: ", 1)[1].split(" | ", 1)[1] for gk in cells})
    all_pairs = [(a, b) for a in cells for b in cells
                 if a != b and len(q_per_cell[a] & q_per_cell[b]) >= MIN_SHARED_Q]
    pick = rng.choice(len(all_pairs), size=min(N_PAIRS, len(all_pairs)), replace=False)
    pairs = [all_pairs[k] for k in pick]
    pair_questions = {}
    for (a, b) in pairs:
        shared = sorted(q_per_cell[a] & q_per_cell[b])
        wds = sorted(((real_wd(a, b, qk), qk) for qk in shared), reverse=True)
        pair_questions[(a, b)] = [qk for _, qk in wds[:N_QUESTIONS]]
    needed = sorted({(gk, qk) for (a, b), qs in pair_questions.items()
                     for qk in qs for gk in (a, b)})
    plan[ty] = dict(cells=sorted({c for p in pairs for c in p}), pairs=pairs,
                    pair_questions=pair_questions, needed=needed,
                    v1_opts=v1_opts, v2_opts=v2_opts)
    print(f"[{ty}] {len(plan[ty]['cells'])} cells, {len(pairs)} pairs, unique baselines {len(needed)}")


## 2. Demographic-QA prompt + identity token positions & opinion-answer position

Same as notebook 12 (identity = 1 answer token in the demographic QA block),
PLUS a new position: **the prompt's last token** (exactly the logits
position already used to read the opinion answer) -- this is what finding 10
uses as the "model state right before answering", to train the probe on.


In [ ]:
ATTR_QA = {
    "RACExRELIG":       ("What is this survey respondent's race?",
                         "What is this survey respondent's religion?"),
    "RELIGxPOLPARTY":   ("What is this survey respondent's religion?",
                         "What is this survey respondent's political party affiliation?"),
    "AGExPOLPARTY":     ("What is this survey respondent's age group?",
                         "What is this survey respondent's political party affiliation?"),
}
DEMO_LETTERS = [chr(65 + i) for i in range(26)]
LETTERS = ["A", "B", "C", "D", "E", "F"]

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def demo_block(question, opts, value):
    lines = [f"Question: {question}"]
    for i, o in enumerate(opts):
        lines.append(f"{DEMO_LETTERS[i]}. {o}")
    lines.append(f"Answer: {DEMO_LETTERS[opts.index(value)]}")
    return "\n".join(lines)

def build_prompt(ty, gk, qk):
    v1, v2 = gk.split(" :: ", 1)[1].split(" | ", 1)
    q1, q2 = ATTR_QA[ty]
    p = plan[ty]
    question, options, _ = qmeta[qk]
    blocks = [demo_block(q1, p["v1_opts"], v1), "", demo_block(q2, p["v2_opts"], v2), "",
              f"Question: {question}"]
    for i, opt in enumerate(options):
        blocks.append(f"{LETTERS[i]}) {opt}")
    blocks.append("Answer:")
    return "\n".join(blocks)

def full_positions(prompt):
    """Positions of the 2 identity tokens + the LAST token (opinion answer)."""
    positions = []
    search_from = 0
    for _ in range(2):
        idx = prompt.find("Answer:", search_from)
        assert idx != -1
        prefix = prompt[: idx + len("Answer:")]
        pos = 1 + len(tokenizer.encode(prefix, add_special_tokens=False))  # +1 BOS
        positions.append(pos)
        search_from = idx + 1
    last_pos = 1 + len(tokenizer.encode(prompt, add_special_tokens=False)) - 1
    positions.append(last_pos)
    return positions  # [id_pos1, id_pos2, last_pos]

# quick check: A vs B differ only at the 2 identity tokens; positions (last_pos included) match
ty0 = TYPES_RUN[0]
(A0, B0) = plan[ty0]["pairs"][0]
qk0 = plan[ty0]["pair_questions"][(A0, B0)][0]
pA, pB = build_prompt(ty0, A0, qk0), build_prompt(ty0, B0, qk0)
tA = tokenizer(pA, return_tensors="pt")["input_ids"][0]
tB = tokenizer(pB, return_tensors="pt")["input_ids"][0]
posA, posB = full_positions(pA), full_positions(pB)
assert len(tA) == len(tB), (len(tA), len(tB))
assert posA == posB, (posA, posB)
assert posA[-1] == len(tA) - 1, "last_pos must equal the index of the last token"
diff = (tA != tB).nonzero().flatten().tolist()
assert set(diff).issubset(set(posA[:2])), "differing token is not at an identity position!"
print("Positions validated:", posA, "(2 identity + 1 opinion-answer position)")


## 3. Load model + batch-index-aware hook engine

Different from notebook 12: patching is now **per-item-in-batch** (each
item may be patched at a different layer), not just globally per layer.
Verified via an offline dry-run before this notebook was written.


In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, torch_dtype=torch.float16, device_map="balanced", low_cpu_mem_usage=True
)
model.eval()
NUM_LAYERS = model.config.num_hidden_layers
NUM_HEADS = model.config.num_attention_heads
HEAD_DIM = model.config.hidden_size // NUM_HEADS
ALL_HEADS = list(range(NUM_HEADS))

LETTER_IDS = [tokenizer.encode(f" {L}", add_special_tokens=False)[-1] for L in LETTERS]
assert len(set(LETTER_IDS)) == len(LETTER_IDS)

rng_ctrl = np.random.default_rng(RANDOM_SEED + 7)
while True:
    RAND_HEAD = (int(rng_ctrl.integers(0, NUM_LAYERS)), int(rng_ctrl.integers(0, NUM_HEADS)))
    if RAND_HEAD not in _forbidden_ctrl:
        break
print("Random control head:", RAND_HEAD)

CAPTURE_LAYERS = list(range(NUM_LAYERS))  # all layers -- needed for the finding 09 sweep donors

_donor_capture = {}   # layer -> {pos: tensor[B, hidden]} (fp16, saves RAM)
_capture_positions = []
_active_patch = {}    # layer -> list of (batch_idx, pos, heads, alpha, donor_vec4096)

def _oproj_prehook(layer_idx):
    def fn(module, args):
        x = args[0]
        if _capture_positions:
            _donor_capture[layer_idx] = {
                pos: x[:, pos, :].detach().half().cpu() for pos in _capture_positions
            }
        patches = _active_patch.get(layer_idx)
        if patches:
            x = x.clone()
            for (b, pos, heads, alpha, donor) in patches:
                d = donor.to(x.device, x.dtype)
                if len(heads) == NUM_HEADS:
                    # optimization: "all heads" = the whole vector -> 1 operation,
                    # not a 32x loop (that is what made the sweep slow: thousands
                    # of separate tiny kernels, though mathematically identical)
                    x[b, pos, :] = x[b, pos, :] + alpha * (d - x[b, pos, :])
                else:
                    for h in heads:
                        s = slice(h * HEAD_DIM, (h + 1) * HEAD_DIM)
                        x[b, pos, s] = x[b, pos, s] + alpha * (d[s] - x[b, pos, s])
            return (x,) + tuple(args[1:])
        return None
    return fn

handles = [model.model.layers[L].self_attn.o_proj.register_forward_pre_hook(_oproj_prehook(L))
           for L in CAPTURE_LAYERS]
print(f"Hooks installed on {len(CAPTURE_LAYERS)} layers.")

@torch.no_grad()
def forward_batch(prompts, n_opt, capture_positions=None, patch_spec=None):
    """One forward for many prompts (identical length). Can capture AND/OR patch."""
    global _capture_positions
    _capture_positions = capture_positions or []
    _active_patch.clear()
    if patch_spec:
        _active_patch.update(patch_spec)
    inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)
    logits = model(**inputs).logits[:, -1, :]
    _capture_positions = []
    _active_patch.clear()
    selected = logits[:, LETTER_IDS[:n_opt]].float()
    return torch.softmax(selected, dim=1).cpu().numpy()


## 4. Pass 1 — baseline (batched per question) + donors for all layers + probe features

Batching follows the notebook 12 pattern exactly (group by `qk`, merge all
`gk` into 1 batch -- same prompt length, differing only in 1-2 identity
letter tokens). Donors for ALL 32 layers are kept in memory (used by the
sweep below, NOT written to disk -- too large). Probe features (the L11
vector at the opinion-answer position) are taken now and stored separately
(small, for finding 10).


In [ ]:
BASELINE_BATCH_SIZE = 16

baseline_pred = {}
donors = {}      # (ty, gk, qk) -> {layer: {pos: vec}}  (in memory, for the sweep)
id_pos = {}      # (ty, qk) -> [id_pos1, id_pos2, last_pos]
probe_rows = []  # (ty, gk, qk, n_opt, mouth_pred) -- metadata for finding 10
probe_vecs = []  # 4096-dim L11 vector at last_pos, aligned with probe_rows

for ty in TYPES_RUN:
    p = plan[ty]
    by_qk = {}
    for (gk, qk) in p["needed"]:
        by_qk.setdefault(qk, []).append(gk)

    for qk, gks in tqdm(by_qk.items(), desc=f"baseline {ty}"):
        prompts = [build_prompt(ty, gk, qk) for gk in gks]
        if (ty, qk) not in id_pos:
            id_pos[(ty, qk)] = full_positions(prompts[0])
        positions = id_pos[(ty, qk)]
        n_opt = len(qmeta[qk][2])
        lens = [len(tokenizer.encode(pr, add_special_tokens=False)) for pr in prompts]

        if len(set(lens)) != 1:
            print(f"  [{ty}/{qk}] token lengths not uniform {set(lens)} -> skip (rare)")
            continue

        for start in range(0, len(gks), BASELINE_BATCH_SIZE):
            batch_gks = gks[start:start + BASELINE_BATCH_SIZE]
            batch_prompts = prompts[start:start + BASELINE_BATCH_SIZE]
            preds = forward_batch(batch_prompts, n_opt, capture_positions=positions)
            for b, gk in enumerate(batch_gks):
                baseline_pred[(gk, qk)] = preds[b]
                donors[(ty, gk, qk)] = {
                    L: {pos: _donor_capture[L][pos][b].clone() for pos in positions}
                    for L in CAPTURE_LAYERS
                }
                last_pos = positions[-1]
                vec_l11 = donors[(ty, gk, qk)][STAR_LAYER][last_pos].float().numpy()
                probe_rows.append(dict(ty=ty, gk=gk, qk=qk, n_opt=n_opt,
                                       mouth_pred=",".join(f"{x:.6f}" for x in preds[b])))
                probe_vecs.append(vec_l11)

print(f"{len(baseline_pred)} baselines done. {len(probe_rows)} probe features collected.")


## 5. Finding 09 — sweep of 32 layers + 1 random control, MERGED into 1 batch

Per (pair, question): the batch holds 33 copies of prompt A, each item
patched at **1 different layer** (all 32 heads in that layer) using donor B;
the 33rd item is patched at 1 random head (control). One forward pass -> 33
conditions at once.


In [ ]:
def wd(pred, real, ordinal):
    return wasserstein_distance(ordinal, ordinal, u_weights=pred, v_weights=real)

sweep_rows = []
for ty in TYPES_RUN:
    p = plan[ty]
    for (A, B) in tqdm(p["pairs"], desc=f"sweep {ty}"):
        for qk in p["pair_questions"][(A, B)]:
            if (ty, qk) not in id_pos or (ty, A, qk) not in donors or (ty, B, qk) not in donors:
                continue
            question, options, ordinal = qmeta[qk]
            n_opt = len(ordinal)
            positions = id_pos[(ty, qk)][:2]  # 2 identity positions (not last_pos)
            realA, realB = real_resp[(A, qk)], real_resp[(B, qk)]
            predA, predB = baseline_pred[(A, qk)], baseline_pred[(B, qk)]
            prompt_A = build_prompt(ty, A, qk)

            n_cond = NUM_LAYERS + 1  # 32 layers + 1 random control
            batch_prompts = [prompt_A] * n_cond
            spec = {}
            for L in range(NUM_LAYERS):
                spec.setdefault(L, [])
                for pos in positions:
                    spec[L].append((L, pos, ALL_HEADS, 1.0, donors[(ty, B, qk)][L][pos]))
            ctrl_idx = NUM_LAYERS
            rl, rh = RAND_HEAD
            spec.setdefault(rl, [])
            for pos in positions:
                spec[rl].append((ctrl_idx, pos, [rh], 1.0, donors[(ty, B, qk)][rl][pos]))

            preds = forward_batch(batch_prompts, n_opt, patch_spec=spec)

            # computed ONCE per (pair, question) -- layer-independent, so do
            # not repeat it 32x inside the L loop below (pointless)
            wd_ctrl_realB = wd(preds[ctrl_idx], realB, ordinal)
            wd_A_realA = wd(predA, realA, ordinal)
            wd_A_realB = wd(predA, realB, ordinal)
            wd_B_realB = wd(predB, realB, ordinal)
            for L in range(NUM_LAYERS):
                sweep_rows.append(dict(
                    attr_type=ty, pair=f"{A} -> {B}", qkey=qk, layer=L,
                    wd_A_to_realA=wd_A_realA,
                    wd_A_to_realB=wd_A_realB,
                    wd_B_to_realB=wd_B_realB,
                    wd_patch_to_realB=wd(preds[L], realB, ordinal),
                    wd_ctrl_to_realB=wd_ctrl_realB,
                ))

sweep = pd.DataFrame(sweep_rows)
sweep["shift_to_realB"] = sweep["wd_A_to_realB"] - sweep["wd_patch_to_realB"]
sweep["shift_ctrl_to_realB"] = sweep["wd_A_to_realB"] - sweep["wd_ctrl_to_realB"]
sweep.to_csv(os.path.join(OUT_DIR, "sweep_rows.csv"), index=False)
print(sweep.shape, "-> sweep_rows.csv")


## 6. Sweep summary: which layer is most influential, per type

Compare each layer against the random control (within the SAME pair-question,
so a paired Wilcoxon) -- main focus: **for RACExRELIG, which layer
"takes over" the role of L11?**


In [ ]:
summary_rows = []
print("=" * 100)
for ty in TYPES_RUN:
    s_ty = sweep[sweep["attr_type"] == ty]
    print(f"\n{ty} -- top 5 layers (mean shift_to_realB):")
    per_layer = s_ty.groupby("layer").agg(
        mean_shift=("shift_to_realB", "mean"),
        pct_pos=("shift_to_realB", lambda x: (x > 0).mean()),
        n=("shift_to_realB", "size"),
    ).reset_index()
    for L in per_layer["layer"]:
        row = s_ty[s_ty["layer"] == L]
        ctrl = row["shift_ctrl_to_realB"]
        if len(row) > 10 and not np.allclose(row["shift_to_realB"], ctrl):
            p = float(wilcoxon(row["shift_to_realB"], ctrl).pvalue)
        else:
            p = float("nan")
        summary_rows.append(dict(attr_type=ty, layer=int(L),
                                 mean_shift=float(row["shift_to_realB"].mean()),
                                 pct_positive=float((row["shift_to_realB"] > 0).mean()),
                                 p_vs_random_ctrl=p))
    top5 = per_layer.sort_values("mean_shift", ascending=False).head(5)
    print(top5.to_string(index=False))

summary = pd.DataFrame(summary_rows)
summary.to_csv(os.path.join(OUT_DIR, "sweep_summary.csv"), index=False)
print("\n-> sweep_summary.csv (all layers x types, with p vs the random control)")


## 7. Save the probe features (finding 10) -- L11 vector at the opinion-answer position

Stored compactly: metadata + the mouth's predictions to CSV, the 4096-dim
vector to a separate npz (to keep the CSV small). Probe fitting is done LOCALLY.


In [ ]:
probe_df = pd.DataFrame(probe_rows)
probe_df.to_csv(os.path.join(OUT_DIR, "probe_features_L11.csv"), index=False)
np.savez_compressed(os.path.join(OUT_DIR, "probe_features_L11.npz"),
                    vecs=np.stack(probe_vecs).astype(np.float32))
print(probe_df.shape, "-> probe_features_L11.csv + probe_features_L11.npz")
print(f"HEAD_DIM={HEAD_DIM}, STAR_HEAD={STAR_HEAD} -> column slice "
      f"[{STAR_HEAD*HEAD_DIM}:{(STAR_HEAD+1)*HEAD_DIM}] of vecs for the 1-head probe.")


## How to read the results & checklist

**Finding 09 (sweep):**
- Look for the layer with the highest `mean_shift` + `p_vs_random_ctrl` <
  0.05 per type. For AGE, the expectation is that L11 (or nearby) is still
  one of the best (cross-validation with finding 08). For **RACE**, look for
  a DIFFERENT winning layer -- that answers "where does the model answer
  from, if not L11".
- If NO layer is significant for RACE -- that is a finding too: not the
  wrong location, but rather race identity genuinely is not used causally in
  any attention (perhaps via the MLP, or perhaps it is not used at all).

**Finding 10 (probe, done locally after downloading):**
- Train a ridge regression per question: `vecs` (or just the head-16 slice)
  -> the REAL distribution, leave-one-group-out cross-validation (small n per
  question). Score the probe's WD vs `mouth_pred` (already in the CSV),
  aggregated per type. Also compare the 32-head probe vs the 1-head probe
  (starhead).

**Download (MANDATORY):** `sweep_rows.csv`, `sweep_summary.csv`,
`probe_features_L11.csv`, `probe_features_L11.npz` ->
`results/07_sweep_and_probe/`.
